In [11]:
# ==============================================================================
# CÉLULA 1: Importação de Bibliotecas e Criação do Dataset
# ==============================================================================
import pandas as pd

# Transcrevendo os dados da imagem 2 para um DataFrame do Pandas
dados = {
    'Idade': ['infantil', 'infantil', 'infantil', 'infantil', 'adolescente', 'adolescente', 'adolescente', 'adolescente', 'adolescente', 'adulto', 'adulto', 'adulto', 'adulto', 'adulto', 'adulto'],
    'Diagnóstico': ['miopia', 'miopia', 'hipermetropia', 'hipermetropia', 'miopia', 'miopia', 'miopia', 'hipermetropia', 'hipermetropia', 'miopia', 'miopia', 'miopia', 'hipermetropia', 'hipermetropia', 'hipermetropia'],
    'Astigmatismo': ['não', 'sim', 'não', 'sim', 'não', 'sim', 'não', 'não', 'sim', 'não', 'sim', 'sim', 'não', 'sim', 'não'],
    'Taxa lacrimal': ['reduzida', 'normal', 'normal', 'normal', 'reduzida', 'reduzida', 'normal', 'reduzida', 'normal', 'normal', 'normal', 'normal', 'reduzida', 'normal', 'normal'],
    'Tipo de lente': ['nenhuma', 'gelatinosa', 'gelatinosa', 'dura', 'gelatinosa', 'nenhuma', 'dura', 'gelatinosa', 'dura', 'gelatinosa', 'dura', 'gelatinosa', 'nenhuma', 'gelatinosa', 'gelatinosa']
}

df = pd.DataFrame(dados)
print("--- Dataset carregado com sucesso! Tamanho:", len(df), "registros.")
display(df.head())


# ==============================================================================
# CÉLULA 2: Exercício 1 - Tabela de Contagens e Probabilidades (Naive Bayes)
# ==============================================================================
print("\n--- EXERCÍCIO 1: Probabilidades Suavizadas Naive Bayes (Laplace alpha=1) ---")

alfa = 1
N = len(df)
classes = df['Tipo de lente'].unique()
num_classes = len(classes)

# Contagem de valores únicos por atributo (para o denominador de Laplace)
v_counts = {col: df[col].nunique() for col in df.columns[:-1]}

# 1. Calculando Probabilidades a Priori P(Classe) com suavização
priors = {}
for c in classes:
    count_c = len(df[df['Tipo de lente'] == c])
    priors[c] = (count_c + alfa) / (N + alfa * num_classes)

# 2. Calculando as Verossimilhanças P(Atributo | Classe) com suavização
likelihoods = {}
for c in classes:
    df_c = df[df['Tipo de lente'] == c]
    count_c = len(df_c)
    likelihoods[c] = {}

    for col in df.columns[:-1]:
        likelihoods[c][col] = {}
        for val in df[col].unique():
            count_val = len(df_c[df_c[col] == val])
            # Fórmula de Laplace: (contagem + alfa) / (total_classe + alfa * num_valores_unicos_do_atributo)
            likelihoods[c][col][val] = (count_val + alfa) / (count_c + alfa * v_counts[col])

print("\nProbabilidades a Priori P(Classe):")
for c, p in priors.items(): print(f"  {c}: {p:.4f}")

print("\n(As tabelas completas de verossimilhança foram armazenadas em memória para os cálculos).")


# ==============================================================================
# CÉLULA 3: Exercício 2 - Previsão do Paciente com Naive Bayes
# ==============================================================================
print("\n--- EXERCÍCIO 2: Previsão Naive Bayes para o novo paciente ---")

paciente = {
    'Idade': 'infantil',
    'Diagnóstico': 'hipermetropia',
    'Astigmatismo': 'não',
    'Taxa lacrimal': 'reduzida'
}

scores_nb = {}
# Calcula o score não-normalizado para cada classe
for c in classes:
    score = priors[c]
    for col, val in paciente.items():
        score *= likelihoods[c][col][val]
    scores_nb[c] = score

# Normaliza para que a soma das probabilidades seja 1 (100%)
soma_scores_nb = sum(scores_nb.values())
probs_norm_nb = {c: s / soma_scores_nb for c, s in scores_nb.items()}

print("\nProbabilidades Normalizadas (Naive Bayes):")
for c, p in probs_norm_nb.items():
    print(f"  {c.upper()}: {p:.4f} ({p*100:.2f}%)")

classe_predita_nb = max(probs_norm_nb, key=probs_norm_nb.get)
print(f" Classificação final Naive Bayes: {classe_predita_nb.upper()}")


# ==============================================================================
# CÉLULA 4: Exercício 3 - Rede Bayesiana (Dependência Idade -> Taxa Lacrimal) 🕸️
# ==============================================================================
print("\n--- EXERCÍCIO 3: Estrutura e Tabelas da Rede Bayesiana ---")
print("Estrutura: Classe -> Idade, Diagnóstico, Astigmatismo, Taxa Lacrimal.")
print("E também: Idade -> Taxa Lacrimal.")
print("Logo, Taxa Lacrimal tem DOIS pais: Classe e Idade.")

# Para Idade, Diagnóstico e Astigmatismo as probabilidades não mudam em relação ao Ex 1.
# Precisamos calcular apenas a nova tabela: P(Taxa Lacrimal | Classe, Idade)

likelihoods_bn_taxa = {}
v_taxa = df['Taxa lacrimal'].nunique()

for c in classes:
    likelihoods_bn_taxa[c] = {}
    for idade in df['Idade'].unique():
        likelihoods_bn_taxa[c][idade] = {}
        # Filtrando pelo cruzamento de Classe e Idade
        df_c_idade = df[(df['Tipo de lente'] == c) & (df['Idade'] == idade)]
        count_c_idade = len(df_c_idade)

        for taxa in df['Taxa lacrimal'].unique():
            count_val = len(df_c_idade[df_c_idade['Taxa lacrimal'] == taxa])
            # Laplace Suavizado
            prob = (count_val + alfa) / (count_c_idade + alfa * v_taxa)
            likelihoods_bn_taxa[c][idade][taxa] = prob

print("\n Tabela de Probabilidade Condicional P(Taxa Lacrimal | Classe, Idade) construída e suavizada com sucesso!")


# ==============================================================================
# CÉLULA 5: Exercício 4 - Reclassificação com a Rede Bayesiana
# ==============================================================================
print("\n--- EXERCÍCIO 4: Previsão do Paciente usando a Rede Bayesiana ---")

scores_bn = {}
for c in classes:
    # 1. P(Classe)
    score = priors[c]

    # 2. P(Atributos Independentes | Classe)
    score *= likelihoods[c]['Idade'][paciente['Idade']]
    score *= likelihoods[c]['Diagnóstico'][paciente['Diagnóstico']]
    score *= likelihoods[c]['Astigmatismo'][paciente['Astigmatismo']]

    # 3. P(Taxa Lacrimal | Classe, Idade) -> A nova dependência!
    score *= likelihoods_bn_taxa[c][paciente['Idade']][paciente['Taxa lacrimal']]

    scores_bn[c] = score

# Normalizando os resultados
soma_scores_bn = sum(scores_bn.values())
probs_norm_bn = {c: s / soma_scores_bn for c, s in scores_bn.items()}

print("\nProbabilidades Normalizadas (Rede Bayesiana):")
for c, p in probs_norm_bn.items():
    print(f"  {c.upper()}: {p:.4f} ({p*100:.2f}%)")

classe_predita_bn = max(probs_norm_bn, key=probs_norm_bn.get)
print(f" Nova classificação final (Rede Bayesiana): {classe_predita_bn.upper()}")

--- Dataset carregado com sucesso! Tamanho: 15 registros.


,Idade,Diagnóstico,Astigmatismo,Taxa lacrimal,Tipo de lente
0,infantil,miopia,não,reduzida,nenhuma
1,infantil,miopia,sim,normal,gelatinosa
2,infantil,hipermetropia,não,normal,gelatinosa
3,infantil,hipermetropia,sim,normal,dura
4,adolescente,miopia,não,reduzida,gelatinosa



--- EXERCÍCIO 1: Probabilidades Suavizadas Naive Bayes (Laplace alpha=1) ---

Probabilidades a Priori P(Classe):
  nenhuma: 0.2222
  gelatinosa: 0.5000
  dura: 0.2778

(As tabelas completas de verossimilhança foram armazenadas em memória para os cálculos).

--- EXERCÍCIO 2: Previsão Naive Bayes para o novo paciente ---

Probabilidades Normalizadas (Naive Bayes):
  NENHUMA: 0.4956 (49.56%)
  GELATINOSA: 0.4276 (42.76%)
  DURA: 0.0768 (7.68%)
 Classificação final Naive Bayes: NENHUMA

--- EXERCÍCIO 3: Estrutura e Tabelas da Rede Bayesiana ---
Estrutura: Classe -> Idade, Diagnóstico, Astigmatismo, Taxa Lacrimal.
E também: Idade -> Taxa Lacrimal.
Logo, Taxa Lacrimal tem DOIS pais: Classe e Idade.

 Tabela de Probabilidade Condicional P(Taxa Lacrimal | Classe, Idade) construída e suavizada com sucesso!

--- EXERCÍCIO 4: Previsão do Paciente usando a Rede Bayesiana ---

Probabilidades Normalizadas (Rede Bayesiana):
  NENHUMA: 0.4474 (44.74%)
  GELATINOSA: 0.3861 (38.61%)
  DURA: 0.1665 (16.